# Generate the inductive-bias-conv dataset

Builds time-series segmentation dataset used by `inductive_bias_conv.ipynb`.

In [ ]:
import numpy as np

## Data definition

In [ ]:
T = 200  # total timesteps per sample
N_SIGNALS = 3

BASELINE_MEAN_RANGE = (-1.0, 1.0)
BASELINE_SIGMAS = [0.75, 0.75, 0.05]

PATTERN_A_SHIFT = np.array([1.8, -1.8])   # added to signals 0 and 1 inside an A-marked B segment

# pattern B oscillation: cycles per timestep
PATTERN_B_FREQ_RANGE = (0.05, 0.25)
PATTERN_B_AMP_RANGE = (0.4, 0.9)

SEGMENT_LEN_RANGE = (30, 80)  # length of one class segment

def _fill_segment(rng, L, cls, has_A=False, has_B=False, t_offset=0):
    # one contiguous segment of length L
    means = rng.uniform(*BASELINE_MEAN_RANGE, size=N_SIGNALS)
    x = means[:, None] + np.stack([rng.normal(0, sigma, size=(L)) for sigma in BASELINE_SIGMAS], axis=0)
    if cls == 1:
        if has_A:
            x[0] += PATTERN_A_SHIFT[0]
            x[1] += PATTERN_A_SHIFT[1]
        if has_B:
            freq = rng.uniform(*PATTERN_B_FREQ_RANGE)
            amp = rng.uniform(*PATTERN_B_AMP_RANGE)
            phase = rng.uniform(0, 2 * np.pi)
            t = np.arange(L) + t_offset
            # resample signal_2 mean with constrained range so mean ± amp stays inside baseline range
            lo, hi = BASELINE_MEAN_RANGE
            bound = max(0.0, min(hi, -lo) - amp)
            m2 = rng.uniform(-bound, bound)
            x[2] = m2 + rng.normal(0, BASELINE_SIGMAS[-1], size=L)
            x[2] += amp * np.sin(2 * np.pi * freq * t + phase)
    return x, np.full(L, cls, dtype=int)

def _b_pattern_flags(rng, p_A=0.8, p_B=0.9):
    a = rng.random() < p_A
    b = rng.random() < p_B
    while not (a or b):
        a = rng.random() < p_A
        b = rng.random() < p_B
    return a, b

def make_sample(rng, p_A=0.8, p_B=0.9, force_pattern=None):
    # build one (3,T) sample plus per-timestep labels.
    # force_pattern in {None, 'A', 'B', 'AB'} forces every class-B segment to that mix.
    xs, ys = [], []
    cls = int(rng.integers(0, 2))           # start randomly with A or B
    t_cursor = 0
    while t_cursor < T:
        L = int(rng.integers(*SEGMENT_LEN_RANGE))
        L = min(L, T - t_cursor)
        if cls == 1:
            if force_pattern is None:
                hA, hB = _b_pattern_flags(rng, p_A, p_B)
            else:
                hA = "A" in force_pattern
                hB = "B" in force_pattern
            x_seg, y_seg = _fill_segment(rng, L, 1, has_A=hA, has_B=hB, t_offset=t_cursor)
        else:
            x_seg, y_seg = _fill_segment(rng, L, 0, t_offset=t_cursor)
        xs.append(x_seg)
        ys.append(y_seg)
        t_cursor += L
        cls = 1 - cls
    X = np.concatenate(xs, axis=1)
    y = np.concatenate(ys)
    return X, y

def make_dataset(n, seed, force_pattern=None):
    rng = np.random.default_rng(seed)
    Xs, Ys = [], []
    for _ in range(n):
        X, y = make_sample(rng, force_pattern=force_pattern)
        Xs.append(X); Ys.append(y)
    return np.stack(Xs), np.stack(Ys)

N_train, N_test = 60, 40
X_train, Y_train = make_dataset(N_train, seed=0)
X_test, Y_test = make_dataset(N_test, seed=1)

diag = {
    "only_A": make_dataset(20, seed=10, force_pattern="A"),
    "only_B": make_dataset(20, seed=11, force_pattern="B"),
    "both":   make_dataset(20, seed=12, force_pattern="AB"),
}

payload = {
    "X_train": X_train, "Y_train": Y_train,
    "X_test":  X_test,  "Y_test":  Y_test,
    "T": np.int64(T), "N_SIGNALS": np.int64(N_SIGNALS),
}
for k, (Xd, Yd) in diag.items():
    payload[f"diag_{k}_X"] = Xd
    payload[f"diag_{k}_Y"] = Yd

np.savez("conv_data.npz", **payload)